# 06 - MLflow Model Registry & Monitoring

**Objective:** Extend MLflow from experiment tracking into production — register models in the
Model Registry, manage versions through staging/production transitions, and build a monitoring
pipeline that detects data drift and model degradation using the bike sharing data injection system.

**Concepts Covered:**
- MLflow Tracking Deep-Dive (MlflowClient, programmatic comparison)
- MLflow Model Registry (registration, versioning, stage transitions)
- Population Stability Index (PSI) for prediction drift (regression binned variant)
- Feature distribution drift (Kolmogorov-Smirnov test)
- Schema drift detection at inference time
- Champion/Challenger deployment pattern
- Automated monitoring loop concept

> **Prerequisite:** This notebook builds on the models trained in `05_model_tuning_validation_comparison.ipynb`.

In [ ]:
import pandas as pd
import numpy as np
from pathlib import Path
import json
import datetime
import warnings
warnings.filterwarnings("ignore")

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR
from xgboost import XGBRegressor
from scipy.stats import ks_2samp

import mlflow
from mlflow.tracking import MlflowClient
import mlflow.sklearn

import matplotlib.pyplot as plt
import seaborn as sns
import joblib

sns.set_style("whitegrid")
plt.rcParams["figure.figsize"] = (10, 6)

print("All imports successful")

In [ ]:
# TODO: Configure MLflow tracking
#
# 1. Define PROJECT_ROOT = Path("..").resolve()
# 2. Create MLFLOW_DIR = PROJECT_ROOT / "mlflow_artifacts" and mkdir
# 3. Set TRACKING_URI = str(MLFLOW_DIR.absolute())
# 4. mlflow.set_tracking_uri(TRACKING_URI)
# 5. mlflow.set_experiment("bike_sharing_06_registry")

PROJECT_ROOT = Path("..").resolve()

# YOUR CODE HERE

In [ ]:
# TODO: Load data, split, scale features
#
# PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
# MODELS_DIR = PROJECT_ROOT / "models"
# df = pd.read_csv(PROCESSED_DIR / "clean_data.csv")
# TARGET = "Rented Bike Count"
# X = df.drop(columns=[TARGET])
# y = df[TARGET]
# FEATURE_NAMES = X.columns.tolist()
#
# X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# joblib.dump(scaler, MODELS_DIR / "scaler_monitoring.joblib")
#
# print(f"Train: {X_train.shape}, Test: {X_test.shape}")

# YOUR CODE HERE

---
## Section 1: Train & Log Models with MLflow

Train the 4 regressors (LinearRegression, RandomForest, XGBoost, SVR) and log each
as an MLflow run. LinearRegression and SVR use **scaled** features; tree models use raw.

In [ ]:
# TODO: Train and log baseline models
#
# configs = [
#     ("LinearRegression", LinearRegression(), True),
#     ("RandomForest", RandomForestRegressor(n_estimators=100, random_state=42), False),
#     ("XGBoost", XGBRegressor(n_estimators=100, random_state=42, eval_metric="rmse"), False),
#     ("SVR", SVR(kernel="rbf"), True),
# ]
# run_ids = {}
#
# mlflow.set_tracking_uri(TRACKING_URI)
# mlflow.set_experiment("bike_sharing_06_registry")
#
# for name, model, use_scaled in configs:
#     X_tr = X_train_scaled if use_scaled else X_train
#     X_te = X_test_scaled if use_scaled else X_test
#     model.fit(X_tr, y_train)
#     preds = model.predict(X_te)
#     mae = mean_absolute_error(y_test, preds)
#     rmse = np.sqrt(mean_squared_error(y_test, preds))
#     r2 = r2_score(y_test, preds)
#
#     with mlflow.start_run(run_name=name) as run:
#         mlflow.log_param("model", name)
#         mlflow.log_metric("mae", mae)
#         mlflow.log_metric("rmse", rmse)
#         mlflow.log_metric("r2", r2)
#         mlflow.sklearn.log_model(model, "model")
#         run_ids[name] = run.info.run_id
#         print(f"{name:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  run_id={run.info.run_id[:8]}...")

# YOUR CODE HERE

---
## Section 2: MLflow Tracking Deep-Dive

Use `MlflowClient` to query experiments programmatically, extract run data into a DataFrame,
and compare models with bar plots.

In [ ]:
# TODO: Query runs with MlflowClient
#
# client = MlflowClient(tracking_uri=TRACKING_URI)
# experiment = client.get_experiment_by_name("bike_sharing_06_registry")
# runs = client.search_runs(experiment_ids=[experiment.experiment_id], order_by=["metrics.r2 DESC"])
#
# for run in runs:
#     name = run.data.tags.get("mlflow.runName", "unnamed")
#     mae = run.data.metrics.get("mae")
#     rmse = run.data.metrics.get("rmse")
#     r2 = run.data.metrics.get("r2")
#     print(f"{name:20s}  MAE={mae:.2f}  RMSE={rmse:.2f}  R2={r2:.4f}  run_id={run.info.run_id[:8]}...")

# YOUR CODE HERE

In [ ]:
# TODO: Extract runs → DataFrame → bar plot
#
# rows = []
# for run in runs:
#     rows.append({
#         "model": run.data.tags.get("mlflow.runName", "unnamed"),
#         "mae": run.data.metrics.get("mae"),
#         "rmse": run.data.metrics.get("rmse"),
#         "r2": run.data.metrics.get("r2"),
#         "run_id": run.info.run_id[:8],
#     })
# run_df = pd.DataFrame(rows)
# display(run_df)
#
# fig, axes = plt.subplots(1, 3, figsize=(15, 5))
# for ax, metric in zip(axes, ["mae", "rmse", "r2"]):
#     sns.barplot(data=run_df, x="model", y=metric, ax=ax, palette="viridis")
#     ax.set_title(f"{metric.upper()} by Model")
#     ax.set_xlabel("")
# plt.tight_layout()
# plt.show()

# YOUR CODE HERE

---
## Section 3: MLflow Model Registry

The **Model Registry** groups models under a registered name, assigns versions,
and manages stages: `None → Staging → Production → Archived`.

> Note: We load by version number (`models:/name/1`), not stage alias (`/Production`),
> because file-based MLflow tracking doesn't support stage-based loading.

In [ ]:
# TODO: Register models in the Model Registry
#
# REGISTERED_MODEL_NAME = "bike_sharing_regressor"
# for name, run_id in run_ids.items():
#     result = mlflow.register_model(f"runs:/{run_id}/model", REGISTERED_MODEL_NAME)
#     print(f"Registered {name:20s} → version {result.version}")

# YOUR CODE HERE

In [ ]:
# TODO: Set version stages with MlflowClient
#
# versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
# run_to_version = {v.run_id: v for v in versions}
#
# # Best R2 → Production
# best_version = run_to_version[runs[0].info.run_id]
# client.transition_model_version_stage(
#     name=REGISTERED_MODEL_NAME, version=best_version.version, stage="Production")
# print(f"v{best_version.version} ({runs[0].data.tags['mlflow.runName']}) → Production")
#
# # Second best → Staging
# second_version = run_to_version[runs[1].info.run_id]
# client.transition_model_version_stage(
#     name=REGISTERED_MODEL_NAME, version=second_version.version, stage="Staging")
# print(f"v{second_version.version} ({runs[1].data.tags['mlflow.runName']}) → Staging")
#
# # Re-fetch versions BEFORE archiving (stale reference bug avoidance)
# versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
# for v in versions:
#     if v.current_stage not in ("Production", "Staging"):
#         client.transition_model_version_stage(
#             name=REGISTERED_MODEL_NAME, version=v.version, stage="Archived")
#         print(f"v{v.version} → Archived")

# YOUR CODE HERE

In [ ]:
# TODO: Load Production model by version number
#
# PRODUCTION_VERSION = best_version.version
# prod_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/{PRODUCTION_VERSION}")
# ref_preds = prod_model.predict(X_test_scaled)
# ref_mae = mean_absolute_error(y_test, ref_preds)
# ref_rmse = np.sqrt(mean_squared_error(y_test, ref_preds))
# ref_r2 = r2_score(y_test, ref_preds)
# print(f"Production model (v{PRODUCTION_VERSION})  MAE={ref_mae:.2f}  RMSE={ref_rmse:.2f}  R2={ref_r2:.4f}")
#
# # Optional: also load and evaluate Staging model
# STAGING_VERSION = second_version.version
# stg_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/{STAGING_VERSION}")
# stg_preds = stg_model.predict(X_test_scaled if runs[1].data.tags.get("mlflow.runName") in ("LinearRegression", "SVR") else X_test)
# stg_mae = mean_absolute_error(y_test, stg_preds)
# print(f"Staging model (v{STAGING_VERSION})  MAE={stg_mae:.2f}")

# YOUR CODE HERE

---
## Section 4: Simulated Monitoring — Data Drift Detection

Evaluate the Production model against 7 pre-generated corrupted variants
(in `data/processed/corrupted/`). Track MAE increase, PSI on binned predictions,
and feature KS-statistics.

In [ ]:
# TODO: Load corrupted datasets
#
# CORRUPTED_DIR = PROCESSED_DIR / "corrupted"
# preset_names = ["missing_light", "missing_heavy", "noise_low", "noise_high",
#                 "outliers", "bias", "schema_drift"]
# corrupted_data = {}
# for preset in preset_names:
#     corrupted_data[preset] = pd.read_csv(CORRUPTED_DIR / f"corrupted_{preset}.csv")
#     print(f"{preset:20s}: {corrupted_data[preset].shape}")

# YOUR CODE HERE

In [ ]:
# TODO: Implement PSI and feature drift functions
#
# def compute_psi(expected, actual, bins=10):
#     """Population Stability Index on binned predictions."""
#     expected = np.clip(expected, 0.001, None)
#     actual = np.clip(actual, 0.001, None)
#     all_vals = np.concatenate([expected, actual])
#     breaks = np.percentile(expected, np.linspace(0, 100, bins + 1))
#     breaks[0] = min(all_vals) - 1e-6
#     breaks[-1] = max(all_vals) + 1e-6
#     ep = np.histogram(expected, breaks)[0] / len(expected)
#     ap = np.histogram(actual, breaks)[0] / len(actual)
#     ep = np.where(ep == 0, 0.001, ep)
#     ap = np.where(ap == 0, 0.001, ap)
#     return np.sum((ap - ep) * np.log(ap / ep))
#
# def compute_feature_drift(ref_df, corr_df, max_cols=5):
#     columns = ref_df.select_dtypes(include=[np.number]).columns[:max_cols]
#     drifts = {}
#     for col in columns:
#         if col in corr_df.columns:
#             stat, pval = ks_2samp(ref_df[col].dropna(), corr_df[col].dropna())
#             drifts[col] = {"ks_stat": stat, "p_value": pval}
#     return drifts

# YOUR CODE HERE

In [ ]:
# TODO: Run drift evaluation for each corrupted preset
#
# mlflow.set_experiment("bike_sharing_06_monitoring")
# monitoring_results = []
#
# for preset_name, corr_df in corrupted_data.items():
#     try:
#         # Detect target column
#         target_col = TARGET if TARGET in corr_df.columns else None
#         if target_col is None:
#             for candidate in ["Rented Bike Count", "bike_count", "rental_count"]:
#                 if candidate in corr_df.columns:
#                     target_col = candidate
#                     break
#
#         if target_col is None:
#             print(f"{preset_name:20s}: SKIP — target column not found")
#             continue
#
#         # Align features: add missing cols with 0, drop extras, reorder
#         X_corr = corr_df.drop(columns=[target_col])
#         for col in FEATURE_NAMES:
#             if col not in X_corr.columns:
#                 X_corr[col] = 0.0
#         X_corr = X_corr[FEATURE_NAMES]
#         y_corr = corr_df[target_col]
#
#         # Drop NaN target rows
#         nan_mask = y_corr.isna()
#         nan_ratio = nan_mask.mean()
#         X_corr = X_corr[~nan_mask]
#         y_corr = y_corr[~nan_mask]
#
#         # Scale features
#         X_corr_scaled = scaler.transform(X_corr)
#
#         # Predict
#         corr_preds = prod_model.predict(X_corr_scaled)
#
#         # Compute metrics
#         mae = mean_absolute_error(y_corr, corr_preds)
#         rmse = np.sqrt(mean_squared_error(y_corr, corr_preds))
#         r2 = r2_score(y_corr, corr_preds)
#         mae_increase = (mae - ref_mae) / ref_mae
#         psi = compute_psi(ref_preds, corr_preds)
#         drift = compute_feature_drift(X_test, X_corr)
#         avg_ks = np.mean([v["ks_stat"] for v in drift.values()]) if drift else 0.0
#
#         monitoring_results.append({
#             "preset": preset_name, "mae": mae, "rmse": rmse, "r2": r2,
#             "mae_increase": mae_increase, "psi": psi, "avg_ks": avg_ks,
#             "nan_target_ratio": nan_ratio,
#         })
#
#         with mlflow.start_run(run_name=preset_name) as run:
#             mlflow.log_metric("mae", mae)
#             mlflow.log_metric("rmse", rmse)
#             mlflow.log_metric("r2", r2)
#             mlflow.log_metric("mae_increase", mae_increase)
#             mlflow.log_metric("psi", psi)
#             mlflow.log_metric("avg_ks", avg_ks)
#             mlflow.log_metric("nan_target_ratio", nan_ratio)
#
#         print(f"{preset_name:20s}  MAE={mae:.2f}  R2={r2:.4f}  MAE_inc={mae_increase:.2%}  PSI={psi:.4f}")
#
#     except Exception as e:
#         print(f"{preset_name:20s}: ERROR — {e}")
#         monitoring_results.append({"preset": preset_name, "mae": None, "rmse": None,
#             "r2": None, "mae_increase": None, "psi": None, "avg_ks": None,
#             "nan_target_ratio": None})

# YOUR CODE HERE

In [ ]:
# TODO: Visualize monitoring results
#
# mon_df = pd.DataFrame(monitoring_results)
# display(mon_df)
#
# fig, axes = plt.subplots(1, 3, figsize=(18, 5))
#
# # MAE increase
# valid = mon_df.dropna(subset=["mae_increase"])
# axes[0].axhline(y=0.20, color="red", linestyle="--", label="Alert threshold (+20%)")
# sns.barplot(data=valid, x="preset", y="mae_increase", ax=axes[0], palette="Reds")
# axes[0].set_title("Relative MAE Increase by Preset")
# axes[0].tick_params(axis="x", rotation=45)
# axes[0].legend()
#
# # PSI
# valid = mon_df.dropna(subset=["psi"])
# axes[1].axhline(y=0.1, color="orange", linestyle=":", label="Moderate (0.1)")
# axes[1].axhline(y=0.25, color="red", linestyle="--", label="Severe (0.25)")
# sns.barplot(data=valid, x="preset", y="psi", ax=axes[1], palette="Blues")
# axes[1].set_title("PSI by Preset")
# axes[1].tick_params(axis="x", rotation=45)
# axes[1].legend()
#
# # Avg KS
# valid = mon_df.dropna(subset=["avg_ks"])
# sns.barplot(data=valid, x="preset", y="avg_ks", ax=axes[2], palette="Greens")
# axes[2].set_title("Avg KS Statistic by Preset")
# axes[2].tick_params(axis="x", rotation=45)
#
# plt.tight_layout()
# plt.show()

# YOUR CODE HERE

In [ ]:
# TODO: Handle schema drift explicitly
#
# schema_df = corrupted_data.get("schema_drift", pd.read_csv(CORRUPTED_DIR / "corrupted_schema_drift.csv"))
# if TARGET in schema_df.columns:
#     print("Target column 'Rented Bike Count' found — no schema drift detected.")
# else:
#     found = [c for c in schema_df.columns if c != TARGET]
#     actual_target = [c for c in found if c in ["bike_count", "rental_count", "target_rented"]]
#     if actual_target:
#         print(f"SCHEMA DRIFT DETECTED: target column renamed from '{TARGET}' to '{actual_target[0]}'")
#         print("Action: update inference pipeline to map column names before serving.")
#     else:
#         print(f"SCHEMA DRIFT DETECTED: target column '{TARGET}' is missing. Columns: {list(schema_df.columns)}")

# YOUR CODE HERE

---
## Section 5: Champion/Challenger Pattern

Train an improved XGBoost challenger, register it, stage it as Staging,
and compare its robustness against the Production champion on corrupted data.

In [ ]:
# TODO: Train a challenger, register, and stage
#
# challenger = XGBRegressor(n_estimators=200, max_depth=7, learning_rate=0.15,
#     random_state=42, eval_metric="rmse")
# challenger.fit(X_train, y_train)
# chall_preds = challenger.predict(X_test)
# chall_mae = mean_absolute_error(y_test, chall_preds)
# print(f"Challenger XGBoost MAE: {chall_mae:.2f}")
#
# with mlflow.start_run(run_name="Challenger_XGBoost") as run:
#     mlflow.log_param("model", "Challenger_XGBoost")
#     mlflow.log_metric("mae", chall_mae)
#     mlflow.sklearn.log_model(challenger, "model")
#     challenger_run_id = run.info.run_id
#
# result = mlflow.register_model(f"runs:/{challenger_run_id}/model", REGISTERED_MODEL_NAME)
# CHALLENGER_VERSION = result.version
# client.transition_model_version_stage(name=REGISTERED_MODEL_NAME,
#     version=CHALLENGER_VERSION, stage="Staging")
# print(f"Challenger v{CHALLENGER_VERSION} → Staging")

# YOUR CODE HERE

In [ ]:
# TODO: Compare champion vs challenger on all corrupted presets
#
# champion_model = prod_model  # already loaded from Registry
# challenger_model = mlflow.sklearn.load_model(f"models:/{REGISTERED_MODEL_NAME}/{CHALLENGER_VERSION}")
#
# comp_rows = []
# for preset_name, corr_df in corrupted_data.items():
#     target_col = TARGET if TARGET in corr_df.columns else (
#         "bike_count" if "bike_count" in corr_df.columns else None)
#     if target_col is None:
#         continue
#     X_c = corr_df.drop(columns=[target_col])
#     for col in FEATURE_NAMES:
#         if col not in X_c.columns:
#             X_c[col] = 0.0
#     X_c = X_c[FEATURE_NAMES]
#     y_c = corr_df[target_col].dropna()
#     X_c = X_c.loc[y_c.index]
#     X_c_scaled = scaler.transform(X_c)
#
#     champ_pred = champion_model.predict(X_c_scaled)
#     chall_pred = challenger_model.predict(X_c)
#
#     champ_mae = mean_absolute_error(y_c, champ_pred)
#     chall_mae = mean_absolute_error(y_c, chall_pred)
#     comp_rows.append({"preset": preset_name,
#         "champion_mae": champ_mae, "challenger_mae": chall_mae,
#         "challenger_wins": chall_mae < champ_mae})
#
# comp_df = pd.DataFrame(comp_rows)
# display(comp_df)
#
# # Grouped bar chart
# comp_df.plot(x="preset", y=["champion_mae", "challenger_mae"], kind="bar",
#     figsize=(10, 5), color=["#2ecc71", "#e74c3c"])
# plt.title("Champion vs Challenger: MAE by Preset")
# plt.ylabel("MAE")
# plt.xticks(rotation=45)
# plt.legend(["Champion (Production)", "Challenger (Staging)"])
# plt.tight_layout()
# plt.show()

# YOUR CODE HERE

In [ ]:
# TODO: Decision gate — promote if challenger wins >50%
#
# wins = comp_df["challenger_wins"].sum()
# total = len(comp_df)
# print(f"Challenger wins on {wins}/{total} scenarios (lower MAE)")
#
# if wins > total / 2:
#     client.transition_model_version_stage(name=REGISTERED_MODEL_NAME,
#         version=CHALLENGER_VERSION, stage="Production")
#     client.transition_model_version_stage(name=REGISTERED_MODEL_NAME,
#         version=PRODUCTION_VERSION, stage="Archived")
#     print("Challenger promoted to Production!")
# else:
#     print("Champion retains Production.")
#
# # Print final registry state
# versions = client.search_model_versions(f"name='{REGISTERED_MODEL_NAME}'")
# for v in versions:
#     print(f"  v{v.version} — stage={v.current_stage}")

# YOUR CODE HERE

---
## Section 6: Automated Monitoring Loop (Conceptual)

Simulate a 5-day monitoring timeline. Each day brings a different corruption type.
Track MAE and PSI over time, and trigger alerts when MAE increase exceeds 20%.

In [ ]:
# TODO: Simulate daily monitoring over 5 days
#
# mlflow.set_experiment("bike_sharing_06_daily_monitoring")
# daily_presets = ["missing_light", "noise_low", "noise_high", "outliers", "bias"]
# timeline = []
#
# for day, preset in enumerate(daily_presets, start=1):
#     corr_df = corrupted_data[preset]
#     target_col = TARGET if TARGET in corr_df.columns else "bike_count"
#
#     X_c = corr_df.drop(columns=[target_col])
#     for col in FEATURE_NAMES:
#         if col not in X_c.columns:
#             X_c[col] = 0.0
#     X_c = X_c[FEATURE_NAMES]
#     y_c = corr_df[target_col].dropna()
#     X_c = X_c.loc[y_c.index]
#     X_c_scaled = scaler.transform(X_c)
#
#     preds = champion_model.predict(X_c_scaled)
#     mae = mean_absolute_error(y_c, preds)
#     rmse = np.sqrt(mean_squared_error(y_c, preds))
#     mae_inc = (mae - ref_mae) / ref_mae
#     psi = compute_psi(ref_preds, preds)
#     alert = mae_inc > 0.20
#
#     with mlflow.start_run(run_name=f"Day_{day}_{preset}") as run:
#         mlflow.log_metric("mae", mae)
#         mlflow.log_metric("rmse", rmse)
#         mlflow.log_metric("mae_increase", mae_inc)
#         mlflow.log_metric("psi", psi)
#
#     timeline.append({"day": day, "preset": preset, "mae": mae,
#         "mae_increase": mae_inc, "psi": psi, "alert": alert})
#     print(f"Day {day} ({preset:15s}): MAE={mae:.2f}  increase={mae_inc:.2%}  PSI={psi:.4f}  alert={alert}")
#
# timeline_df = pd.DataFrame(timeline)
# display(timeline_df)
#
# # Dual-axis plot
# fig, ax1 = plt.subplots(figsize=(10, 5))
# ax1.plot(timeline_df["day"], timeline_df["mae"], "o-", color="#2ecc71", linewidth=2, label="MAE")
# ax1.axhline(y=ref_mae, color="green", linestyle=":", label="Baseline MAE")
# ax1.axhline(y=ref_mae * 1.2, color="red", linestyle="--", label="Alert threshold (+20%)")
# ax1.set_xlabel("Day")
# ax1.set_ylabel("MAE", color="#2ecc71")
# ax1.tick_params(axis="y", labelcolor="#2ecc71")
# ax1.set_xticks(timeline_df["day"])
#
# ax2 = ax1.twinx()
# ax2.plot(timeline_df["day"], timeline_df["psi"], "s--", color="#e74c3c", linewidth=2, label="PSI")
# ax2.set_ylabel("PSI", color="#e74c3c")
# ax2.tick_params(axis="y", labelcolor="#e74c3c")
#
# lines1, labels1 = ax1.get_legend_handles_labels()
# lines2, labels2 = ax2.get_legend_handles_labels()
# ax1.legend(lines1 + lines2, labels1 + labels2, loc="upper left")
#
# # Highlight alert days
# alert_days = timeline_df[timeline_df["alert"]]["day"]
# for d in alert_days:
#     ax1.axvspan(d - 0.3, d + 0.3, alpha=0.15, color="red")
#
# plt.title("Daily Monitoring: MAE and PSI over Time")
# plt.tight_layout()
# plt.show()

# YOUR CODE HERE

---
## Summary

1. **MlflowClient API**: Query experiments, extract runs, compare programmatically
2. **Model Registry**: Register models, assign versions, transition through stages
3. **Drift Monitoring**: PSI on binned predictions, KS-test on features, MAE increase thresholds
4. **Schema Drift**: Detecting column mismatches at inference time
5. **Champion/Challenger**: Systematic model comparison and promotion decisions
6. **Automated Monitoring**: Scheduled evaluation loops with alerting

---
## Exercises

1. **Alerting System**: Define `check_alerts(monitoring_df, mae_threshold=0.20, psi_threshold=0.25)`
   that returns triggered alerts. How many presets trigger each alert type?

2. **Multi-Model Monitoring**: Register each model as separate registered names
   (e.g. `bike_sharing_lr`, `bike_sharing_xgb`). Which is most robust to drift?

3. **Feature Importance Drift**: For champion and challenger (tree-based), extract
   feature importances. Are top-5 features the same?

4. **Custom Corruption**: Add a new preset to `data_injection/config.py` simulating
   seasonal drift, regenerate corrupted data, re-run monitoring.

5. **Auto-Retraining**: Build a function that monitors → detects drift → auto-retrains
   → registers → compares → promotes if better. Self-healing ML pipeline.